# Chapter 12 — Custom Models and Training with TensorFlow
Companion notebook to `part2_deep_learning/12_custom_models_tensorflow.md`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

housing = fetch_california_housing()
X_train_full, X_test, y_train_full, y_test = train_test_split(housing.data, housing.target, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_full, y_train_full, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_valid = scaler.transform(X_valid).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)
y_train, y_valid, y_test = y_train.astype(np.float32), y_valid.astype(np.float32), y_test.astype(np.float32)

## Custom loss function: Huber loss, from scratch

In [ ]:
def huber_fn(y_true, y_pred):
    error = y_true - y_pred
    is_small_error = tf.abs(error) < 1
    squared_loss = tf.square(error) / 2
    linear_loss = tf.abs(error) - 0.5
    return tf.where(is_small_error, squared_loss, linear_loss)

model = keras.models.Sequential([
    keras.layers.Dense(30, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.Dense(1),
])
model.compile(loss=huber_fn, optimizer="nadam")
model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

## Custom layer with trainable weights

In [ ]:
class MyDense(keras.layers.Layer):
    def __init__(self, units, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.activation = keras.activations.get(activation)

    def build(self, batch_input_shape):
        self.kernel = self.add_weight(
            name="kernel", shape=[batch_input_shape[-1], self.units],
            initializer="glorot_normal")
        self.bias = self.add_weight(name="bias", shape=[self.units], initializer="zeros")
        super().build(batch_input_shape)

    def call(self, X):
        return self.activation(X @ self.kernel + self.bias)

custom_model = keras.models.Sequential([
    MyDense(30, activation="relu", input_shape=X_train.shape[1:]),
    MyDense(1),
])
custom_model.compile(loss="mse", optimizer="nadam")
custom_model.fit(X_train, y_train, epochs=5, validation_data=(X_valid, y_valid))

## tf.GradientTape: computing gradients by hand

In [ ]:
w1, w2 = tf.Variable(5.), tf.Variable(3.)

def f(w1, w2):
    return 3 * w1 ** 2 + 2 * w1 * w2

with tf.GradientTape() as tape:
    z = f(w1, w2)
gradients = tape.gradient(z, [w1, w2])
print("Gradients:", [g.numpy() for g in gradients])

## A custom training loop

In [ ]:
model2 = keras.models.Sequential([
    keras.layers.Dense(30, activation="elu", kernel_initializer="he_normal",
                        input_shape=X_train.shape[1:]),
    keras.layers.Dense(1),
])

def random_batch(X, y, batch_size=32):
    idx = np.random.randint(len(X), size=batch_size)
    return X[idx], y[idx]

n_epochs, batch_size = 5, 32
n_steps = len(X_train) // batch_size
optimizer = keras.optimizers.Nadam(learning_rate=0.01)
loss_fn = keras.losses.mean_squared_error
mean_loss = keras.metrics.Mean()

for epoch in range(1, n_epochs + 1):
    for step in range(1, n_steps + 1):
        X_batch, y_batch = random_batch(X_train, y_train, batch_size)
        with tf.GradientTape() as tape:
            y_pred = model2(X_batch, training=True)
            main_loss = tf.reduce_mean(loss_fn(y_batch, tf.squeeze(y_pred)))
            loss = tf.add_n([main_loss] + model2.losses)
        gradients = tape.gradient(loss, model2.trainable_variables)
        optimizer.apply_gradients(zip(gradients, model2.trainable_variables))
        mean_loss(loss)
    print(f"Epoch {epoch}: mean loss = {mean_loss.result():.4f}")
    mean_loss.reset_states()

## ✍️ Practice
1. Implement a `log_cosh` custom loss and compare its behavior on outlier-heavy data vs MSE.
2. Why must a custom loss return one value per instance, not an averaged scalar?
3. Write a custom layer implementing a trainable weighted skip connection: `alpha*input + (1-alpha)*Dense(units)(input)`.
4. Debug: in a custom training loop, BatchNormalization always behaves as if in inference mode — what line is missing?

In [ ]:
# Your code / notes here